In [1]:

import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

data_path = Path("../data/processed_churn_data.csv")

data = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", data.shape)
print("Columns:", data.columns.tolist())

display(data.head())


Dataset loaded successfully!
Dataset shape: (7043, 26)
Columns: ['City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'CLTV']


,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,CLTV
0,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,3239
1,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2701
2,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,5372
3,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,5003
4,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,5340


In [3]:

target_column = "Churn Label"

X = data.drop(columns=[target_column])
y = data[target_column]

# Convert Yes/No to 1/0 if the target contains text
if y.dtype == "object":
    y = y.map({"No": 0, "Yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())


Features shape: (7043, 25)
Target shape: (7043,)

Target values:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (5634, 25)
Testing data: (1409, 25)


In [5]:

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [13]:

param_grid = {
    "model__C": [0.01, 0.1, 1],
    "model__solver": ["liblinear"],
    "model__penalty": ["l1", "l2"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    error_score="raise"
)

print("GridSearchCV is ready!")


GridSearchCV is ready!


In [8]:

target_column = "Churn Label"

X = data.drop(columns=[target_column]).copy()

# Convert churn labels into numeric values
y = data[target_column].map({
    "No": 0,
    "Yes": 1
})

# Check that conversion worked
print("Target values:")
print(y.value_counts(dropna=False))

print("Missing target values:", y.isna().sum())


Target values:
Churn Label
0    5174
1    1869
Name: count, dtype: int64
Missing target values: 0


In [10]:

print("y values:", y.unique())
print("y_train values:", y_train.unique())
print("Target column values:", data["Churn Label"].unique())


y values: [1 0]
y_train values: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Target column values: <StringArray>
['Yes', 'No']
Length: 2, dtype: str


In [11]:

# Convert target labels to numeric values
y = data["Churn Label"].astype(str).str.strip().map({
    "No": 0,
    "Yes": 1
})

# Check for unexpected labels
if y.isna().any():
    raise ValueError(
        "Unexpected target labels found. Check data['Churn Label'].unique()."
    )

X = data.drop(columns=["Churn Label"]).copy()

# Create a fresh train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training target labels:", y_train.unique())
print("Testing target labels:", y_test.unique())
print("Target conversion successful!")


Training target labels: [0 1]
Testing target labels: [0 1]
Target conversion successful!


In [15]:

from sklearn.metrics import make_scorer, f1_score

# Explicitly tell GridSearchCV that 1 means churn (Yes)
f1_scorer = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

param_grid = {
    "model__C": [0.01, 0.1, 1],
    "model__solver": ["liblinear"],
    "model__penalty": ["l1", "l2"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

print("GridSearchCV is ready!")


GridSearchCV is ready!


In [16]:

from sklearn.metrics import make_scorer, f1_score

# Explicitly tell GridSearchCV that 1 means churn (Yes)
f1_scorer = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

param_grid = {
    "model__C": [0.01, 0.1, 1],
    "model__solver": ["liblinear"],
    "model__penalty": ["l1", "l2"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

print("GridSearchCV is ready!")


GridSearchCV is ready!


In [17]:

print("Training labels:", y_train.unique())
print("Testing labels:", y_test.unique())


Training labels: [0 1]
Testing labels: [0 1]


In [19]:

import warnings
from sklearn.exceptions import ConvergenceWarning

# Hide the specific warnings from Logistic Regression
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Test different regularization strengths
param_grid = {
    "model__C": [0.01, 0.1, 1],
    "model__solver": ["liblinear"],
    "model__l1_ratio": [0, 1]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

print("GridSearchCV is ready without the deprecated penalty parameter!")


GridSearchCV is ready without the deprecated penalty parameter!


In [20]:
grid_search.fit(X_train, y_train)

print("Tuning completed!")
print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Tuning completed!
Best parameters: {'model__C': 1, 'model__l1_ratio': 1, 'model__solver': 'liblinear'}
Best CV F1-score: 0.6194


In [21]:

# Train and tune the model
grid_search.fit(X_train, y_train)

print("Tuning completed!")
print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Tuning completed!
Best parameters: {'model__C': 1, 'model__l1_ratio': 1, 'model__solver': 'liblinear'}
Best CV F1-score: 0.6194


In [22]:

# Get the best model
best_model = grid_search.best_estimator_

# Predict on test data
y_pred_tuned = best_model.predict(X_test)

print("Predictions completed!")


Predictions completed!


In [23]:

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, zero_division=0)
recall_tuned = recall_score(y_test, y_pred_tuned, zero_division=0)
f1_tuned = f1_score(y_test, y_pred_tuned, zero_division=0)

tuned_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Tuned Model": [
        accuracy_tuned,
        precision_tuned,
        recall_tuned,
        f1_tuned
    ]
})

display(tuned_results.round(4))


,Metric,Tuned Model
0,Accuracy,0.8006
1,Precision,0.6388
2,Recall,0.5722
3,F1-score,0.6037


In [24]:

from pathlib import Path
import joblib

# Create the models folder if it doesn't exist
models_folder = Path("../models")
models_folder.mkdir(parents=True, exist_ok=True)

# Save the best tuned model
model_path = models_folder / "best_logistic_regression.pkl"
joblib.dump(best_model, model_path)

print("Best model saved successfully!")
print("Saved location:", model_path.resolve())


Best model saved successfully!
Saved location: C:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\models\best_logistic_regression.pkl


In [25]:

baseline_results = {
    "Accuracy": 0.7743,
    "Precision": 0.5952,
    "Recall": 0.4679,
    "F1-score": 0.5240
}

comparison = pd.DataFrame({
    "Metric": list(baseline_results.keys()),
    "Baseline Model": list(baseline_results.values()),
    "Tuned Model": [
        accuracy_tuned,
        precision_tuned,
        recall_tuned,
        f1_tuned
    ]
})

comparison["Improvement"] = (
    comparison["Tuned Model"] - comparison["Baseline Model"]
)

display(comparison.round(4))


,Metric,Baseline Model,Tuned Model,Improvement
0,Accuracy,0.7743,0.8006,0.0263
1,Precision,0.5952,0.6388,0.0436
2,Recall,0.4679,0.5722,0.1043
3,F1-score,0.5240,0.6037,0.0797


## Conclusion

GridSearchCV was used to tune the Logistic Regression model by testing different hyperparameter combinations. The best model was evaluated using accuracy, precision, recall, and F1-score.

The tuned model was compared with the baseline model to check whether its performance improved. The best tuned model was saved as `best_logistic_regression.pkl` in the `models` folder.
